# NTEMS

## Download Data

In [ ]:
import wget
import os
import zipfile


def download_and_unzip(url, output_directory):
    filename = wget.download(url, out=output_directory)
    print(f"\nDownload complete. File saved as: {filename}")
    # Unzip the file
    zip_file_path = os.path.join(output_directory, f"CA_Tree_Species_Classification_{year}.zip")
    with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall(output_directory)
    print(f"Files extracted successfully to: {output_directory}")

prefix = "https://opendata.nfis.org/downloads/forest_change"
urls = [
    f"{prefix}/CA_forest_harvest_mask_year_1985_2015.zip",
    f"{prefix}/CA_forest_wildfire_year_DNBR_Magnitude_1985_2015.zip",
]
years = ["2018", "2019", "2020", "2021", "2022"]
for year in years:
    urls.append(f"{prefix}/CA_Tree_Species_Classification_{year}.zip")

output_directory = "/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems"
os.makedirs(output_directory, exist_ok=True)  # Create directory if it doesn't exist
for url in urls:
    download_and_unzip(url, output_directory)

## Generate Disturbance Dataset

In [ ]:
import glob
from os.path import join
import geopandas
import rasterio
from image_utils import clip_raster, merge_rasters

directory=r'/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/ON/disturb'
boundary = r"/mnt/d/Sync/research/tree_species_estimation/tree_dataset/data_processing/FORMGMT/LIO-2023-08-19/FOREST_MANAGEMENT_UNIT.shp"
fire_1985_2020 = r'/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/CA_Forest_Fire_1985-2020.tif'
harvest_1985_2020 = r'/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/CA_Forest_Harvest_1985-2020.tif'

with rasterio.open(fire_1985_2020) as src:
    out_crs = src.crs

gdf = geopandas.read_file(boundary).to_crs(out_crs)
reproject_boundary = r"/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/ON/ROI_ON_ntems.shp"
gdf.to_file(reproject_boundary)

clip_raster(
    fire_1985_2020,
    reproject_boundary,
    join(directory, "on_forest_fire_1985_2020_clipped.tif"),
)
clip_raster(
    harvest_1985_2020,
    reproject_boundary,
    join(directory, "on_forest_harvest_1985_2020_clipped.tif"),
)

# merge rasters from disturbances of different years
directory = r"/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/ON"
input_files = glob.glob(join(directory, "disturb", "*.tif"))
output_file = join(directory, "disturb", "on_ntems_merged_1985_2020.tif")
merge_rasters(input_files, output_file, method="max")

In [ ]:
# merge disturbances 1985-2020

# Open the raster file
with rasterio.open(
    r"/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/ON/disturb/on_ntems_merged_1985_2020.tif"
) as src:
    # Read the data into an array
    raster_data = src.read(1)  # Assuming the data is in the first band

    # Replace values 1985 to 2020 with 1 (indicating change)
    change_mask = (raster_data >= 1985) & (raster_data <= 2020)
    raster_data[change_mask] = 1

    # Define new metadata for the output file
    new_meta = src.meta.copy()

# Save the modified raster
with rasterio.open(
    r"/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/ON/disturb/on_ntems_merged_years.tif",
    "w",
    **new_meta
) as dst:
    dst.write(raster_data, 1)

## Mask Disturbance

In [ ]:

import os
from os.path import join
from image_utils import ntems_mask

directory = r"/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/ON"
mask_file = join(directory, "disturb", "on_ntems_merged_years.tif")
boundary = os.path.join(directory, "ROI_ON_ntems.gpkg")
# Data codes: 0 = no change, 1 = wildfire, 2 = harvest,
# 5 = lower confidence wildfire, 6 = lower confidence harvest
mask_values = [1]
years = ["2018", "2019", "2020", "2021", "2022"]
for year in years:
    ntems_mask(
        f"/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/ON/ON_Tree_Species_Classification_{year}.tif",
        mask_file,
        mask_values,
        join(directory, f"ON_Tree_Species_Classification_{year}_ntems.tif"),
    )

## Generate Grids

Generate the Stable Species Mask

In [6]:
import rasterio
import numpy as np
import os

def create_stable_mask(ntems_paths, output_mask_path):
    """
    Creates a single .tif where a pixel has a value ONLY if it was
    identical across all input years and not 0 or 34.
    """
    srcs = [rasterio.open(p) for p in ntems_paths]
    meta = srcs[0].meta.copy()
    meta.update(
        dtype="uint8", nodata=0, compress="lzw"
    )  # Use compression to save disk space

    with rasterio.open(output_mask_path, "w", **meta) as dst:
        # Loop through the raster in blocks (e.g., 256x256 or 512x512)
        for _, window in srcs[0].block_windows():
            # Read all years for this block
            stack = np.array([src.read(1, window=window) for src in srcs])

            # Identify pixels where:
            # 1. All years are equal
            # 2. Value is not 0 (non-treed)
            # 3. Value is not 34 (Hemlock)
            stable_mask = (
                np.all(stack == stack[0], axis=0) & (stack[0] != 0) & (stack[0] != 34)
            )

            # Create output: stable species ID if True, else 0
            out_block = np.where(stable_mask, stack[0], 0).astype("uint8")

            dst.write(out_block, window=window, indexes=1)

    for src in srcs:
        src.close()
    print(f"Stable Mask saved to {output_mask_path}")


folder = "/mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/ON"
years = ["2018", "2019", "2020", "2021", "2022"]
ntems_files = []

for year in years:
    ntems_files.append(
        os.path.join(folder, f"ON_Tree_Species_Classification_{year}_ntems.tif")
    )
create_stable_mask(ntems_files, os.path.join(folder, "Ontario_Stable_Mask.tif"))

Stable Mask saved to /mnt/d/Sync/research/tree_species_estimation/tree_dataset/ntems/ON/Ontario_Stable_Mask.tif


In [18]:
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd


def sampled_stable_coords_distributed(mask_path, tempfile, cap=10000, seed=42):
    np.random.seed(seed)

    # Define your specific 15 target species IDs (based on your table)
    valid_species = [2, 5, 6, 9, 10, 12, 13, 17, 18, 22, 25, 26, 28, 29, 31, 32]
    print("Step 1: Counting species distribution (Filtering 255/NoData)...")
    with rasterio.open(mask_path) as src:
        data = src.read(1)
        # Only count if value is in our target list
        mask = np.isin(data, valid_species)
        unique, counts = np.unique(data[mask], return_counts=True)
        stats = dict(zip(unique, counts))

    # Calculate probabilities with a safety buffer
    probs = {sid: min(1.0, (cap * 1.3) / count) for sid, count in stats.items()}
    species_coords = {sid: [] for sid in stats.keys()}

    print("Step 2: Geographically distributed sampling...")
    with rasterio.open(mask_path) as src:
        # Process in blocks to save RAM
        for _, window in src.block_windows():
            block = src.read(1, window=window)

            # Mask block for only valid species (ignores 0, 34, 255)
            rows, cols = np.where(np.isin(block, valid_species))

            for r, c in zip(rows, cols):
                sid = block[r, c]

                # Check if this species still needs samples
                if len(species_coords[sid]) < cap:
                    # Probabilistic skip for geographical spread
                    if np.random.random() < probs[sid]:
                        # FIXED: Use 'c' for column index
                        x, y = rasterio.transform.xy(
                            src.transform,
                            r + window.row_off,
                            c + window.col_off,
                            offset="center",
                        )
                        species_coords[sid].append({"x": x, "y": y, "label": int(sid)})

    # Flatten and final trim
    all_records = [item for sublist in species_coords.values() for item in sublist]
    df = pd.DataFrame(all_records)

    # Final trim to exactly the cap
    df = df.groupby("label").head(cap).reset_index(drop=True)

    print("\n--- Final Balanced Sample Counts ---")
    print(df["label"].value_counts().sort_index())
    df.to_csv(tempfile)

In [19]:
def finalize_pretrain_plan(tempcsv, spl_index_shp, ecoregion_shp, output_gpkg):
    df_coords = pd.read_csv(tempcsv)
    print(f"samples input: {len(df_coords)} plots.")
    # Convert points to GeoDataFrame (NTEMS is 3978)
    gdf = gpd.GeoDataFrame(
        df_coords, geometry=gpd.points_from_xy(df_coords.x, df_coords.y), crs=3978
    )

    # 1. Join with Ecoregions
    ecoregions = gpd.read_file(ecoregion_shp).to_crs(3978)
    gdf = gpd.sjoin(
        gdf, ecoregions[["SITE_REG_O", "geometry"]], how="inner", predicate="within"
    )

    # 2. Join with SPL Index
    spl_index = gpd.read_file(spl_index_shp).to_crs(3978)
    # This identifies the 1km tile each plot belongs to
    final_gdf = gpd.sjoin(
        gdf,
        spl_index[["Tilename", "Download_H", "geometry"]],
        how="inner",
        predicate="within",
        rsuffix="_spl",
    )

    # 3. Save to GPKG
    final_gdf.to_file(output_gpkg, driver="GPKG", layer="sampling_plan_10k")
    print(f"Final sampling plan created: {len(final_gdf)} plots.")

In [20]:
aux_folder = "/mnt/d/Sync/research/tree_species_estimation/tree_dataset"
spl_index_path = os.path.join(aux_folder, "data_processing", "FRI_Leaf_On_Tile_Index_SHP", "FRI_Tile_Index.shp")
ecoregion_path = os.path.join(aux_folder, "mapping", "Mapping", "EcoRegion", "Site_Region_.shp")
tempcsv = os.path.join(folder, "ON_Pretrain_SPL_cap10k_temp.csv")
output_gpkg = os.path.join(folder, "ON_Pretrain_SPL_cap10k.gpkg")

sampled_stable_coords_distributed(os.path.join(folder, "Ontario_Stable_Mask.tif"), tempcsv)
finalize_pretrain_plan(tempcsv, spl_index_path, ecoregion_path, output_gpkg)

Step 1: Counting species distribution (Filtering 255/NoData)...
Step 2: Geographically distributed sampling...

--- Final Balanced Sample Counts ---
label
2     10000
5     10000
6     10000
9        78
10    10000
12    10000
13     1466
17     8634
18    10000
22    10000
25    10000
26    10000
28     8323
29    10000
31    10000
32    10000
Name: count, dtype: int64
samples input: 138501 plots.
Final sampling plan created: 126581 plots.


# download dataset

In [ ]:
import subprocess
import os
from pathlib import Path

# --- TEST SETTINGS ---
# Use a real URL from your GPKG here
test_url = "https://download.fri.mnrf.gov.on.ca/api/api/Download/hag/utm18/1kmZ183260504102022L_HAG.copc.laz"
# ---------------------


def test_download():
    # 1. Determine local path
    # We use current directory '.' for this local test
    filename = test_url.split("/")[-1]
    local_path = Path(".").resolve() / filename

    print(f"Targeting download to: {local_path}")

    # 2. Try Download
    # Remove -q so we can see the errors!
    cmd = ["wget", "-O", str(local_path), test_url]
    print(f"Running command: {' '.join(cmd)}")

    result = subprocess.run(cmd, capture_output=True, text=True)

    if local_path.exists():
        print(f"✅ SUCCESS: File exists at {local_path}")
        print(f"📏 SIZE: {local_path.stat().st_size / 1e6:.2f} MB")
    else:
        print(f"❌ FAILED: File does not exist.")
        print(f"STDOUT: {result.stdout}")
        print(f"STDERR: {result.stderr}")


if __name__ == "__main__":
    test_download()

Targeting download to: /mnt/d/Sync/research/tree_species_estimation/code/CE-TSC/data_processing/1kmZ183260504102022L_HAG.copc.laz
Running command: wget -O /mnt/d/Sync/research/tree_species_estimation/code/CE-TSC/data_processing/1kmZ183260504102022L_HAG.copc.laz https://download.fri.mnrf.gov.on.ca/api/api/Download/hag/utm18/1kmZ183260504102022L_HAG.copc.laz
✅ SUCCESS: File exists at /mnt/d/Sync/research/tree_species_estimation/code/CE-TSC/data_processing/1kmZ183260504102022L_HAG.copc.laz
📏 SIZE: 314.03 MB


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import pdal
from pathlib import Path
from pyproj import Transformer


def debug_local_processing(laz_path, gpkg_path, tile_name_to_test):
    print(f"--- Starting Diagnostic for Tile: {tile_name_to_test} ---")

    # 1. Load your GPKG and filter for this specific tile
    gdf = gpd.read_file(gpkg_path, layer="sampling_plan_10k")
    tile_plots = gdf[gdf["Tilename"] == tile_name_to_test]

    if len(tile_plots) == 0:
        print(f"Error: No plots found in GPKG for tile {tile_name_to_test}")
        return

    # 2. Get Tile Metadata (Native EPSG and Bounding Box)
    pipeline = pdal.Pipeline(
        json.dumps(
            [
                {"type": "readers.las", "filename": str(laz_path)},
                {"type": "filters.stats"},
            ]
        )
    )
    pipeline.execute()
    metadata = []

    # 4. Test the FIRST plot in this tile
    row = tile_plots.iloc[0]
    cx_tile, cy_tile = row["x"], row["y"]

    # 5. Run PDAL Crop
    p_json = [
        {"type": "readers.las", "filename": str(laz_path)},
        {
            "type": "filters.crop",
            "point": f"POINT({cx_tile} {cy_tile})",
            "distance": 11.28,
        },
        {"type": "filters.range", "limits": "Z(2:)"},
    ]

    try:
        pipe = pdal.Pipeline(json.dumps(p_json))
        pipe.execute()
        pts = pipe.arrays[0]
        print(f"Success! Found {len(pts)} points above 2m.")

        if len(pts) >= 7168:
            # Test Save
            out_dir = Path("./debug_out")
            out_dir.mkdir(exist_ok=True)
            test_path = out_dir / "test_plot.npy"

            # Center and Sample
            data = np.vstack((pts["X"] - cx_tile, pts["Y"] - cy_tile, pts["Z"])).T
            idx = np.random.choice(data.shape[0], 7168, replace=False)
            np.save(str(test_path), data[idx])

            if test_path.exists():
                print(f"IO Check: SUCCESS. File saved to {test_path.absolute()}")
            else:
                print("IO Check: FAILED. np.save did not write file.")
        else:
            print(
                f"Filter Check: FAILED. Not enough points (found {len(pts)}, need 7168)."
            )

    except Exception as e:
        print(f"PDAL Pipeline Check: FAILED with error: {e}")

In [5]:
# HOW TO RUN:
debug_local_processing(
    "D:\\Sync\\research\\tree_species_estimation\\tree_dataset\\ntems\\ON\\1kmZ183260504102022L_HAG.copc.laz",
    "D:\\Sync\\research\\tree_species_estimation\\tree_dataset\\ntems\\ON\\ON_Pretrain_SPL_cap10k.gpkg",
    "1kmZ183260504102022L",
)

<>:3: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.
<>:4: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.
<>:3: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.
<>:4: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.
C:\Users\ycao68\AppData\Local\Temp\ipykernel_26784\4100242510.py:3: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.
  "D:\\Sync\\research\\tree_species_estimation\\tree_dataset\\ntems\ON\\1kmZ183260504102022L_HAG.copc.laz",
C:\Users\ycao68\AppData\Local\Temp\ipykernel_26784\4100242510.py:

--- Starting Diagnostic for Tile: 1kmZ183260504102022L ---


C:\Users\ycao68\AppData\Local\Temp\ipykernel_26784\4100242510.py:3: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.
  "D:\\Sync\\research\\tree_species_estimation\\tree_dataset\\ntems\ON\\1kmZ183260504102022L_HAG.copc.laz",
C:\Users\ycao68\AppData\Local\Temp\ipykernel_26784\4100242510.py:4: SyntaxWarning: "\O" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\O"? A raw string is also an option.
  "D:\\Sync\\research\\tree_species_estimation\\tree_dataset\\ntems\ON\\ON_Pretrain_SPL_cap10k.gpkg",


TypeError: the JSON object must be str, bytes or bytearray, not dict

# Dataset splitting

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Merge all the small CSVs from the Slurm batches
import glob

all_files = glob.glob("meta_batch_*.csv")
df = pd.concat([pd.read_csv(f) for f in all_files])

# 2. Create a combined "Stratify Key"
# This ensures balance across Species AND Ecoregion
df["stratify_key"] = df["label"].astype(str) + "_" + df["ecoregion"].astype(str)

# 3. First Split: Separate Test set (10%)
train_val_df, test_df = train_test_split(
    df, test_size=0.10, stratify=df["stratify_key"], random_state=42
)

# 4. Second Split: Separate Train and Val from the remainder
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.111,
    stratify=train_val_df["stratify_key"],
    random_state=42,
)  # 0.111 of 0.90 is roughly 10% of total

# 5. Save the splits
train_df.to_csv("train_split.csv", index=False)
val_df.to_csv("val_split.csv", index=False)
test_df.to_csv("test_split.csv", index=False)

print(f"Splitting complete!")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")